# 02 - Preprocessing
Cleaning rules, projection choice, motion columns, candidate-search settings and reference-label
quality. Partition used: **training** (student-testing is only used for evaluation).

In [ ]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt

OUT, GRAPHS, MAPS = ROOT / "outputs", ROOT / "outputs" / "graphs", ROOT / "outputs" / "maps"
PROC = ROOT / "dataset" / "processed"
for d in (GRAPHS, MAPS, PROC, ROOT / "models" / "configurations"):
    d.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 200, "display.max_columns", 40)

## 1. Projection check: local metric plane vs haversine

In [ ]:
from src import data_loader as dl
from src.preprocessing import LocalProjection, haversine_m

rng = np.random.default_rng(0)
errs = []
for tid in dl.list_trips("training")[:25]:
    n = dl.load_trip("training", tid, with_route=False).nodes
    proj = LocalProjection(n[:, 0].mean(), n[:, 1].mean())
    i, j = rng.integers(0, len(n), (2, 800))
    x1, y1 = proj.forward(n[i, 0], n[i, 1]); x2, y2 = proj.forward(n[j, 0], n[j, 1])
    d_proj, d_hav = np.hypot(x1 - x2, y1 - y2), haversine_m(n[i, 0], n[i, 1], n[j, 0], n[j, 1])
    ok = d_hav > 50
    errs.append(100 * (d_proj[ok] - d_hav[ok]) / d_hav[ok])
errs = np.abs(np.concatenate(errs))
print(f"abs relative distance error (percent): median {np.median(errs):.4f}  95th pct {np.percentile(errs, 95):.4f}  max {errs.max():.4f}")

## 2. Cleaning report over all training trips

In [ ]:
from src.preprocessing import clean_track
reports = []
for tid in dl.list_trips("training"):
    _, rep = clean_track(dl.load_trip("training", tid, with_route=False).track)
    reports.append(rep)
rep = pd.DataFrame(reports)
print(rep.sum(numeric_only=True)); print("trips not time-sorted:", int((~rep.was_time_sorted).sum()))

## 3. Speed and heading columns (one training trip)

In [ ]:
from src.preprocessing import add_motion_columns
tid = dl.list_trips("training")[10]
trip = dl.load_trip("training", tid, with_route=False)
track, _ = clean_track(trip.track)
proj = LocalProjection(trip.nodes[:, 0].mean(), trip.nodes[:, 1].mean())
mot = add_motion_columns(track, proj)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(mot.t, mot.speed_mps); ax[0].set(title=f"trip {tid}: speed", xlabel="t (s)", ylabel="m/s")
ax[1].plot(mot.t, mot.heading_deg, ".", ms=2); ax[1].set(title="heading (deg from north)", xlabel="t (s)")
fig.tight_layout(); fig.savefig(GRAPHS / "preprocessing_speed_heading.png"); plt.show()
print("share of points with unreliable heading (vehicle ~stationary):", 1 - mot.heading_reliable.mean())

## 4. Candidate search settings: radius and maximum candidates
Candidates are generated with a deliberately generous setting (100 m, 30 candidates) on 15 training trips,
then we measure how often the reference arc would be lost at smaller radius / K.

In [ ]:
from src.feature_engineering import FeatureConfig, build_trip_features
big = FeatureConfig(radius_m=100.0, max_candidates=30)
pos_rows, n_obs_total = [], 0
for tid in dl.list_trips("training")[::5][:15]:
    tf = build_trip_features(dl.load_trip("training", tid), big)
    pos_rows.append(tf.cands[tf.cands.y == 1][["dist_m", "dist_rank"]]); n_obs_total += len(tf.obs)
pos = pd.concat(pos_rows)
radii = [20, 30, 40, 50, 60, 80, 100]; ks = [1, 2, 3, 5, 8, 10, 15, 20, 30]
miss_r = [1 - (pos.dist_m <= r).sum() / n_obs_total for r in radii]
miss_k = [1 - (pos.dist_rank < k).sum() / n_obs_total for k in ks]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(radii, miss_r, "o-"); ax[0].set(xlabel="search radius (m)", ylabel="candidate miss rate", title="Miss rate vs radius")
ax[1].plot(ks, miss_k, "o-"); ax[1].set(xlabel="max candidates per observation", title="Miss rate vs K (radius 100 m)")
fig.tight_layout(); fig.savefig(GRAPHS / "candidate_miss_rate.png"); plt.show()
pd.DataFrame({"radius_m": radii, "miss_rate": miss_r}).round(4)

In [ ]:
print(pd.DataFrame({"K": ks, "miss_rate": miss_k}).round(4))
# The defaults in src/feature_engineering.FeatureConfig (radius 60 m, K = 10) should be justified by the curves above.

## 5. Quality of the reference labels (alignment of GPS points to the route)

In [ ]:
d = pos.dist_m
print("aligned reference arc: distance quantiles (m):", d.quantile([.5, .9, .99]).round(1).to_dict())
print("share of aligned reference arcs that are NOT the nearest candidate: %.3f" % (pos.dist_rank > 0).mean())
print("observations without any aligned arc: %.4f" % (1 - len(pos) / n_obs_total))

## 6. Decisions (fill in)
* Projection:
* Cleaning rules:
* Candidate radius and K:
* Limits of the reference route / alignment: 